In [ ]:
!pip uninstall weaviate-client -y
!pip install weaviate-client==3.26.7
!pip install langchain openai sentence-transformers pypdf tiktoken --upgrade
!pip install -U langchain-community
!pip install -q transformers accelerate bitsandbytes

In [ ]:
import os
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import Weaviate
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import weaviate
from weaviate.auth import AuthApiKey
from huggingface_hub import login

In [ ]:
# 🔐 Login to HuggingFace
login("HUGGING_FACE_KEY")

# 🔐 Set API keys
os.environ["OPENAI_API_KEY"] = "OPENAI_API_KEY"
WEAVIATE_URL = "WEAVIATE_URL"
WEAVIATE_API_KEY = "WEAVIATE_API_KEY"

In [ ]:
# 📘 Load and split the PDF
def process_documents(source_path):
    loader = PyPDFLoader(source_path)
    documents = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    return splitter.split_documents(documents)

In [ ]:
# 🧠 Create vector store and upload to Weaviate
def create_vector_store(docs):
    embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

    client = weaviate.Client(
        url=f"https://{WEAVIATE_URL}",
        auth_client_secret=AuthApiKey(WEAVIATE_API_KEY),
        additional_headers={"X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"]}
    )

    existing_classes = [cls["class"] for cls in client.schema.get()["classes"]]
    if "MyIndex" not in existing_classes:
        schema = {
            "classes": [
                {
                    "class": "MyIndex",
                    "vectorizer": "none",
                    "properties": [{"name": "page_content", "dataType": ["text"]}]
                }
            ]
        }
        client.schema.create(schema)

    texts = [doc.page_content for doc in docs]
    vectors = embeddings.embed_documents(texts)

    for text, vector in zip(texts, vectors):
        client.data_object.create(
            data_object={"page_content": text},
            class_name="MyIndex",
            vector=vector
        )

    vectorstore = Weaviate(
        client=client,
        index_name="MyIndex",
        text_key="page_content",
        embedding=embeddings
    )

    return vectorstore, embeddings, client

In [ ]:
# 💬 Ask a question
def answer_question(client, embeddings, query, pipe):
    CLASS_NAME = "MyIndex"
    query_vector = embeddings.embed_query(query)

    response = client.query.get(
        CLASS_NAME,
        ["page_content"]
    ).with_near_vector({
        "vector": query_vector
    }).with_limit(3).do()  # Fetch top 3 matches

    hits = response["data"]["Get"].get(CLASS_NAME, [])
    if not hits:
        return "Sorry, I couldn't find anything relevant."

    docs_text = "\n\n".join(hit["page_content"] for hit in hits)
    docs_text = docs_text[:1500]  # ✂️ Trim context to 1500 chars

    prompt = f"""### Instruction:
You are a helpful assistant. Use the following context to answer the question thoroughly.

### Context:
{docs_text}

### Question: {query}
### Answer:"""

    try:
        output = pipe(
            prompt,
            max_new_tokens=512,           # 🔼 Increased output length
            do_sample=True,
            temperature=0.7,              # 🔥 Controlled creativity
            top_k=50,
            top_p=0.9
        )[0]["generated_text"]

        return output.split("### Answer:")[-1].strip()
    except Exception as e:
        print("Error:", e)
        return "Sorry, something went wrong while generating the answer."

In [ ]:
import glob
# 🚀 Main
if __name__ == "__main__":
    # 📂 Load all PDFs from folder
    source_folder_path = "/content/Rights"  # Update with your folder path
    pdf_files = glob.glob(os.path.join(source_folder_path, "*.pdf"))

    all_docs = []
    for pdf_file in pdf_files:
        print(f"Processing: {pdf_file}")
        docs = process_documents(pdf_file)
        all_docs.extend(docs)

    # 🧠 Create vector store with combined docs
    vectorstore, embeddings, client = create_vector_store(all_docs)

    # Load Mistral 7B only once
    model_id = "mistralai/Mistral-7B-Instruct-v0.1"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

    # Q&A Loop
    while True:
        user_question = input("\nAsk a question (or type 'exit'): ")
        if user_question.lower() == "exit":
            break
        response = answer_question(client, embeddings, user_question, pipe)
        print("\nAnswer:", response)